In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
train = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/train.csv')
oil = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/oil.csv')
stores = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/stores.csv')
holidays = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/holidays_events.csv')
transactions = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/transactions.csv')
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_log_error

In [ ]:
train.head()

In [ ]:
train = train.drop(columns=["id"])
train.head()

In [ ]:
stores.head()

In [ ]:
oil.head()

In [ ]:
daily_sales = (    
	train 
	.groupby('date')['sales']
	.sum()
	.reset_index())

In [ ]:
df = pd.merge(
    daily_sales, 
    oil[['date','dcoilwtico']], 
    on='date', 
    how='inner'
)

In [ ]:
plt.figure(figsize=(8,6))
sns.scatterplot(
    x='dcoilwtico', 
    y='sales', 
    data=df, 
    alpha=0.6
)
plt.title('Daily Sales vs Oil Price')
plt.xlabel('Oil Price (dcoilwtico)')
plt.ylabel('Daily Total Sales')
plt.grid(True)
plt.show()


In [ ]:
corr = df['sales'].corr(df['dcoilwtico'])
print(f'Pearson 相関係数: {corr:.3f}')

In [ ]:
plt.figure(figsize=(8,6))
sns.regplot(
    x='dcoilwtico', 
    y='sales', 
    data=df, 
    scatter_kws={'alpha':0.4}, 
    line_kws={'color':'red'}
)
plt.title('Sales vs Oil Price with Regression Line')
plt.xlabel('Oil Price (dcoilwtico)')
plt.ylabel('Daily Total Sales')
plt.grid(True)
plt.show()

In [ ]:
holidays.head()

In [ ]:
holidays["is_holiday"] = holidays["type"].isin(["Holiday", "Additional", "Bridge"])

holidays["is_event"] = holidays["type"] == "Event"

holidays.head()

In [ ]:
holidays = holidays.drop(columns=["type"])
holidays.head()

In [ ]:
transactions.head()

In [ ]:
merged_s = pd.merge(train, stores, on="store_nbr",how="left")
merged_so = pd.merge(merged_s, oil, on="date",how="left")
merged_soh = pd.merge(merged_so, holidays, on="date",how="left")
merged = pd.merge(merged_soh, transactions, on=["date","store_nbr"],how="left")

merged.head()

In [ ]:
merged["dcoilwtico"] = merged["dcoilwtico"].fillna(method="ffill").fillna(method="bfill")

merged["transactions"] = merged["transactions"].fillna(0)

merged[["is_holiday", "is_event"]] = merged[["is_holiday", "is_event"]].fillna(False)

merged["transferred"] = merged["transferred"].fillna(False)

merged.head()

In [ ]:
merged["date"] = pd.to_datetime(merged["date"])

merged["year"] = merged["date"].dt.year
merged["month"] = merged["date"].dt.month
merged["day"] = merged["date"].dt.day

merged["dayofweek"] = merged["date"].dt.dayofweek

merged["is_weekend"] = merged["dayofweek"] >= 5

merged.head()

In [ ]:
merged.loc[(merged["locale"] == "Local") & (merged["city"] != merged["locale_name"]),["is_holiday","is_event"]] = False
merged.loc[(merged["locale"] == "Regional") & (merged["state"] != merged["locale_name"]),["is_holiday","is_event"]] = False

merged.head()

In [ ]:
merged = merged.drop(columns=["locale","locale_name"])

merged.head()

In [ ]:
merged = merged.drop(columns=["city","state","description","date"])

merged.head()

In [ ]:
training=pd.get_dummies(merged, columns=["store_nbr","family","type","cluster","dayofweek"], drop_first=True)
training.head()

In [ ]:
from sklearn.preprocessing import StandardScaler
train_standard = StandardScaler()
train_copied = training.copy()
train_standard.fit(train_copied[['year', 'month','day']])
train_std = pd.DataFrame(train_standard.transform(train_copied[['year', 'month','day']]))
train_std.head()

In [ ]:
train_std.columns = ['year_std', 'month_std', 'day_std']
training = training.drop(columns=['year', 'month', 'day'])
training = pd.concat([training, train_std], axis=1)
training.head()

In [ ]:
test = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/test.csv')
test.head()

In [ ]:
merged_ts = pd.merge(test, stores, on="store_nbr",how="left")
merged_tso = pd.merge(merged_ts, oil, on="date",how="left")
merged_tsoh = pd.merge(merged_tso, holidays, on="date",how="left")
merged_test = pd.merge(merged_tsoh, transactions, on=["date","store_nbr"],how="left")

merged_test.head()

In [ ]:
merged_test["dcoilwtico"] = merged_test["dcoilwtico"].fillna(method="ffill").fillna(method="bfill")

merged_test["transactions"] = merged_test["transactions"].fillna(0)

merged_test[["is_holiday", "is_event"]] = merged_test[["is_holiday", "is_event"]].fillna(False)

merged_test["transferred"] = merged_test["transferred"].fillna(False)

merged_test.head()

In [ ]:
merged_test["date"] = pd.to_datetime(merged_test["date"])


merged_test["year"] = merged_test["date"].dt.year
merged_test["month"] = merged_test["date"].dt.month
merged_test["day"] = merged_test["date"].dt.day


merged_test["dayofweek"] = merged_test["date"].dt.dayofweek

merged_test["is_weekend"] = merged_test["dayofweek"] >= 5

merged_test.head()

In [ ]:
merged_test.loc[(merged_test["locale"] == "Local") & (merged_test["city"] != merged_test["locale_name"]),["is_holiday","is_event"]] = False
merged_test.loc[(merged_test["locale"] == "Regional") & (merged_test["state"] != merged_test["locale_name"]),["is_holiday","is_event"]] = False

merged_test.head()

In [ ]:
merged_test = merged_test.drop(columns=["locale","locale_name"])

merged_test.head()

In [ ]:
merged_test = merged_test.drop(columns=["city","state","description","date"])

merged_test.head()

In [ ]:
testing=pd.get_dummies(merged_test, columns=["store_nbr","family","type","cluster","dayofweek"], drop_first=True)
testing.head()

In [ ]:
from sklearn.preprocessing import StandardScaler
test_standard = StandardScaler()
test_copied = testing.copy()
test_standard.fit(test_copied[['year', 'month','day']])
test_std = pd.DataFrame(test_standard.transform(test_copied[['year', 'month','day']]))
test_std.head()

In [ ]:
test_std.columns = ['year_std', 'month_std', 'day_std']
testing = testing.drop(columns=['year', 'month', 'day'])
testing = pd.concat([testing, test_std], axis=1)
testing.head()

In [ ]:
X = training.drop(columns=["sales"])
y = training["sales"]

In [ ]:
X.columns = X.columns.str.replace('[^A-Za-z0-9_]', '_', regex=True)
testing.columns = testing.columns.str.replace('[^A-Za-z0-9_]', '_', regex=True)

In [ ]:
feature_names = X.columns

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, shuffle=False)

In [ ]:
train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_val, label=y_val)

In [ ]:
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'verbose': -1
}

In [ ]:
callbacks = [lgb.early_stopping(stopping_rounds=50)]
model = lgb.train(
    params,
    train_data,
    valid_sets=[train_data, val_data],
    valid_names=['train', 'valid'],
    num_boost_round=1000,
    callbacks=callbacks
)

In [ ]:
y_val_pred = model.predict(X_val)
y_val_pred = np.maximum(0, y_val_pred)  
rmsle = np.sqrt(mean_squared_log_error(y_val, y_val_pred))
print(f"Validation RMSLE: {rmsle:.4f}")

In [ ]:
X_test = testing[feature_names]
y_test_pred = model.predict(X_test)
y_test_pred = np.maximum(0, y_test_pred) 

In [ ]:
submission = pd.DataFrame({
    "id": testing["id"],
    "sales": y_test_pred
})
submission.to_csv("submission.csv", index=False)
print("✅ submission.csv を保存しました")